# Install Dependencies

In [ ]:
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.4/366.4 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 34.4 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets
!pip install -q regex

In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 2.9 MB/s eta 0:00:00


In [ ]:
!pip install -q diffusers

# login

In [ ]:
import huggingface_hub
huggingface_hub.login('HF_TOKEN')

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.utils import shuffle
import os
import re
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             precision_score,
                             recall_score,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
import emoji
import pyarabic.araby as araby

In [ ]:
import pandas as pd

In [ ]:
import torch
import torch.distributed as dist

# Load Model

In [ ]:
model_name = "QCRI/Fanar-1-9B-Instruct"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )

# Assign pad_token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

config.json:   0%|          | 0.00/900 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.69G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/2.12M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/18.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

# Load Data

In [ ]:
import pandas as pd
data = pd.read_excel('sampled_data.xlsx')

In [ ]:
data.head()

,main directory,subdirectory,content
0,Khaleej,Culture,استقبل الوسط المسرحي الإماراتي فوز الإمارات بر...
1,Khaleej,Culture,استضاف مركز الشارقة لفن الخط العربي والزخرفة م...
2,Khaleej,Culture,باسمة يونس قد يبدو العنوان اسماً لرواية؛ لكنه ...
3,Khaleej,Culture,أبوظبي: «الخليج» أكد عدد من الخبراء والمسؤولين...
4,Khaleej,Culture,يمكن القول باطمئنان أن شهر رمضان المبارك هو شه...


# Zero Shot

In [ ]:
content = '''أنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال:
1. الثقافة
2. المال
3. الطب
4. السياسة
5. الدين
6. الرياضة
7. التكنولوجيا
'''

zero_pred = []
for i, text in enumerate(data['content']):
    prompt = f'''المقال:
    {text}

    الفئة المتوقعة:'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=20,
        do_sample=True,
        temperature = 0.2,
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    zero_pred.append(response)

In [ ]:
pred_zero = pd.DataFrame()
pred_zero['Predicted'] = zero_pred
pred_zero['Predicted'].value_counts()

,count
Predicted,
السياسة,170
الرياضة,150
الثقافة,140
التكنولوجيا,133
الطب,129
الدين,90
المال,81
مال,45
المال,23


In [ ]:
pred_zero['Article'] = data['content']
pred_zero.to_excel('Fanar-NC-ZeroShot.xlsx', index = False)

The output is checked for normalizing predictions since the output does not follow a single format

In [ ]:
pred_zero = pd.read_excel('Fanar-NC-ZeroShot.xlsx')

In [ ]:
pred_zero['Predicted Normalized'].value_counts()

,count
Predicted Normalized,
السياسة,174
الرياضة,155
الثقافة,147
المال,144
التكنولوجيا,138
الطب,137
الدين,92
المال,7
Other,6


In [ ]:
nor_pre = []
for pr in pred_zero['Predicted Normalized']:
  if "الرياضة" in pr:
    nor_pre.append("Sports")
  elif "الصحة" in pr:
    nor_pre.append("Medical")
  elif "الطب" in pr:
    nor_pre.append("Medical")
  elif "الثقافة" in pr:
    nor_pre.append("Culture")
  elif "المال" in pr:
    nor_pre.append("Finance")
  elif "السياسة" in pr:
    nor_pre.append("Politics")
  elif "الدين" in pr:
    nor_pre.append("Religion")
  elif "التكنولوجيا" in pr:
    nor_pre.append("Tech")
  else:
    nor_pre.append("Other")

In [ ]:
y_true = data['subdirectory'].values
print(classification_report(y_true, nor_pre, digits = 4))

              precision    recall  f1-score   support

     Culture     0.8639    0.8467    0.8552       150
     Finance     0.8344    0.8400    0.8372       150
     Medical     0.9635    0.8800    0.9199       150
       Other     0.0000    0.0000    0.0000         0
    Politics     0.8046    0.9333    0.8642       150
    Religion     0.9022    0.8300    0.8646       100
      Sports     0.9613    0.9933    0.9770       150
        Tech     0.9203    0.8467    0.8819       150

    accuracy                         0.8840      1000
   macro avg     0.7813    0.7712    0.7750      1000
weighted avg     0.8924    0.8840    0.8868      1000



# Pred Few Shot

In [ ]:
content = '''أنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال:
1. الثقافة
2. المال
3. الطب
4. السياسة
5. الدين
6. الرياضة
7. التكنولوجيا

المثال 1
محتوى المقال:
أكد الفنان عزت أبوعوف، رئيس مهرجان القاهرة السينمائي، إلغاء حفل ختام المهرجان الذي كان من المزمع أن يقام غداً، وذلك بسبب الظروف غير المستقرة التي تمر بها مصر حالياً.ونفى أبوعوف في تصريحات لـ"العربية.نت" سفر الضيوف الأجانب خوفاً مما تشهده مصر من مظاهرات، وما تردد عن أن هذا السفر قد تسبب في عدم إمكانية إقامة حفل الختام، مؤكداً أن وزير الثقافة دكتور محمد صابر عرب هو من أصدر قرار الإلغاء.وعن توزيع الجوائز التي كان من المزمع أن يتم في حفل الختام، شرح أنه سيقام مؤتمر صحافي سيكون بديلاً عن الحفل وسيتم من خلاله إعلان توزيع الجوائز والأفلام الفائزة في هذه الدورة.يُذكر أن حفل افتتاح المهرجان أيضاً تم في هدوء وبعيداً عن الصخب المعتاد، وذلك أيضاً بسبب الظروف نفسها التي تشهدها مصر.

الفئة المتوقعة: الثقافة

المثال 2
محتوى المقال:
قال الرئيس التنفيذي للشركة السعودية للكهرباء زياد الشيحة، في مقابلة عبر الهاتف مع قناة "العربية"، إنه لأول مرة في تاريخ الشركة تراجع استهلاك المملكة في فترات الذروة. وأضاف الشيحة أن التراجع الذي حدث في استهلاك المملكة في 2016، مقارنة مع عام 2015، دفع الشركة لمراجعة السعات المطلوبة لدى دراسة المشاريع الجديدة. وأكد أن مسألة انخفاض الحمل الذروي لأول مرة في تاريخ الشركة عن العام جلعنا نراجع المحطات المستقبلية والتي ستكون بعقود شراء الطاقة، وهذا سيكون لمشاربع الإنتاج وتتم مراجعة السعات المطلوبة خاصة مع قلة الحمل الذروي في 2016. وستزود الشركة السعودية للكهرباء شركة زين السعودية بشبكة الألياف البصرية الممتدة لـ 60 ألف كم. وأوضح الشيحة أن "قطاع التوليد سيطرح للخصخصة كما هو معلن، ونعمل على الموضوع بشكل متوازن وشبه يومي". وكانت خسائر شركة السعودية للكهرباء قد تفاقمت بأكثر من 60%، في الربع الأخير من العام الماضي، مقارنة بالربع المماثل من عام 2015، لتبلغ 2.34 مليار ريال. من ناحية أخرى، ارتفعت أرباح الشركة بنسبة 37%، خلال العام الماضي، مقارنةً بعام 2015، لتبلغ 2.1 مليار ريال. وأرجعت الشركة تفاقم الخسائر الفصلية إلى ارتفاع تكلفة المبيعات نتيجة الزيادة في أسعار الوقود وارتفاع المصاريف التشغيلية.

الفئة المتوقعة: المال


المثال 3
محتوى المقال:
يعتقد بعض المدخنين أن السجائر الإلكترونية تعد أحد أهم العوامل المساعدة في الإقلاع عن التدخين، في حين يعتقد البعض الآخر أنها تعتبر وسيلة إغواء للاستمرار في الخضوع لتلك العادة المدمرة، إلا أن الأبحاث الطبية الحديثة تشير إلى أن الأخطار والفوائد لتلك النوعية من السجائر لاتزال غير معلومة بصورة واضحة بين المدخنين، وذلك وفق ما نشرت وكالة أنباء الشرق الأوسط المصرية. وكانت مجموعة من الباحثين قد أجرت أبحاثها على أكثر من 64 مدخنا، ولم ينجحوا في تحقيق إجماع حول الفوائد والأضرار المحتملة للسجائر الإلكترونية، وهو ما قد يعكس انقساما في المجتمع الطبي حول مدى ملاءمة تعزيز السجائر الإلكترونية كبديل أكثر أمنا للتدخين. وأوضح الباحثون أن معظم المشاركين في الدراسة يرون أن التدخين يعتبر شكلا من الإدمان، حيث تلعب الإرادة دورا قويا في الإقلاع عن هذه العادة المدمرة، في الوقت الذى حاول فيه جميع المشاركين في الدراسة مرة واحدة على الأقل الإقلاع عن العادة المدمرة.

الفئة المتوقعة: الطب


المثال 4
محتوى المقال:
أعرب المتحدث باسم الهيئة العليا للمفاوضات السورية سالم المسلط عن أمله في أن تنتقل روسيا فعليا لتقف إلى جانب الشعب السوري بدلا من النظام، وذلك عقب قراره بسحب القوات الروسية من سوريا. وأضاف المسلط أن هناك جدية لمست مؤخرا حيال المواقف الروسية للدفع نحو الحل السياسي للأزمة في سوريا، خلال جولة المحادثات الجديدة التي انطلقت في جنيف. هذا وأعلن متحدث باسم الرئيس الروسي فلاديمير بوتين بوتين بأن روسيا أبلغت الأسد بقرار سحب الجزء الرئيسي من القوات الروسية من سوريا. وقال المتحدث إن بوتين خلال اجتماعه بوزير دفاعه أمر اعتبارا من اليوم (الثلاثاء) ببدء سحب الجزء الرئيسي من القوات الروسية. في حين قال متحدث باسم بوتين إن القاعدة البحرية والجوية الروسية في سوريا تستمر في العمل كما في السابق. ووفقا للكرملين فإن بوتين طلب من وزير خارجيته سيرغي لافروف تكثيف الدور الروسي في عملية السلام في سوريا، مشيرا إلى ان  القوات الروسية في سوريا أوجدت ظروفا ملائمة لعملية السلام.

الفئة المتوقعة: السياسة


المثال 5
محتوى المقال:
أكد عبداللطيف بخاري، رئيس لجنة الخبراء باتحاد القدم السعودي أن اتحاده يبحث عن 15 منصباً في اللجان الآسيوية، وأن هناك لجنة برئاسة خالد المرزوقي، عضو مجلس إدارة الاتحاد مكلفة بالاختيار. وقال بخاري لـ"في المرمى":" الترشيحات تتم بناء على معايير قارية، ونحن نبحث عن 15 منصباً في لجان الاتحاد الآسيوي". وبين بخاري أن لجنة المسابقات اقترحت إيجاد مراقب لكل مباراة، وزاد:" تقارير المراقب لن تغني عن تقارير الحكم، أما بالنسبة لتقارير الأول فيمكن للجنة الانضباط الاستناد عليها".

الفئة المتوقعة: الرياضة


المثال 6
محتوى المقال:
يبدو أن هاتف #آيفون7  الجديد الذي ستصدره شركة آبل، لن يحمل الكثير من التغيرات، بحسب ما أفاد تقرير لـ "وول ستريت جورنول". فالهاتف الجديد سيأتي شبيها بالنسخة الحالية (آيفون6)، مع تغيير جذري على صعيد "الصوت" والسماعات. وحسب تقرير الصحيفة قد تزيل شركة #آبل منفذ سماعة الصوت، لتدمجه بالمنفذ الذي يوضع فيه شاحن الهاتف في الأسفل. وتراهن آبل من خلال إزالة المنفذ على جعل هاتفها الجديد "أرفع"، ومضاد للماء فإزالة فتحة السماعة، ستمنع تسرب الماء إلى الجهاز عبر الثقب، وتعطيله. ومن شأن إزالة المنفذ الذي يصل قطره إلى 2.5 ميلليمتر أن ينعكس إيجابا أيضاً على البطارية، على اعتبار أن التغيير سيفسح مساحة جديدة يمكن استغلالها. إلا أن التقرير لم يفصل كيف يمكن لمنفذ الشحن الجديد أن يمنع بدوره تسرب الماء إلى داخل الهاتف. في المقابل، يرى بعض منتقدي الشكل الجديد أو التغيير المنتظر أن الاستغناء عن السماعات التقليدية، سيجبر المستخدمين على شراء السماعات الأغلى التي تعمل بتقنية "بلوتوث".

الفئة المتوقعة: التكنولوجيا


المثال 7
محتوى المقال:
} قال رسول الله صلى الله عليه وسلم: «من أتى فراشه وهو ينوي أن يقوم يصلي من الليل فغلبته عينه حتى أصبح، كتب له ما نوى، وكان نومه صدقة عليه من ربه».} وقال صلى الله عليه وسلم: «ما تشاور قومإلا هداهم الله لأرشد أمورهم».

الفئة المتوقعة: الدين
'''

few_pred = []
for i, text in enumerate(data['content']):
    prompt = f'''المقال الذي عليك تصنيفه
    محتوى المقال:
    {text}

    الفئة المتوقعة:'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=20,
        do_sample=True,
        temperature = 0.2,
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    few_pred.append(response)

In [ ]:
pred_few = pd.DataFrame()
pred_few['Predicted'] = few_pred
pred_few['Predicted'].value_counts()

,count
Predicted,
السياسة,158
الثقافة,156
الرياضة,150
الطب,126
التكنولوجيا,119
الدين,80
مال,63
المال,62
المال,34


In [ ]:
pred_few['Article'] = data['content']
pred_few.to_excel('Fanar-NewsClassification-FewShot.xlsx', index = False)

The output is checked manually since ot does not follow a single format

In [ ]:
pred_few = pd.read_excel('Fanar-NewsClassification-FewShot.xlsx')

In [ ]:
pred_few['Predicted Normalized'].value_counts()

,count
Predicted Normalized,
الثقافة,161
المال,160
السياسة,159
الرياضة,151
الطب,139
التكنولوجيا,124
الدين,82
Other,24


In [ ]:
nor_pre = []
for pr in pred_few['Predicted Normalized']:
  if "الرياضة" in pr:
    nor_pre.append("Sports")
  elif "الصحة" in pr:
    nor_pre.append("Medical")
  elif "الطب" in pr:
    nor_pre.append("Medical")
  elif "الثقافة" in pr:
    nor_pre.append("Culture")
  elif "المال" in pr:
    nor_pre.append("Finance")
  elif "السياسة" in pr:
    nor_pre.append("Politics")
  elif "الدين" in pr:
    nor_pre.append("Religion")
  elif "التكنولوجيا" in pr:
    nor_pre.append("Tech")
  else:
    nor_pre.append("Other")

In [ ]:
y_true = data['subdirectory'].values
print(classification_report(y_true, nor_pre, digits = 4))

              precision    recall  f1-score   support

     Culture     0.8509    0.9133    0.8810       150
     Finance     0.8125    0.8667    0.8387       150
     Medical     0.9712    0.9000    0.9343       150
       Other     0.0000    0.0000    0.0000         0
    Politics     0.8616    0.9133    0.8867       150
    Religion     0.9634    0.7900    0.8681       100
      Sports     0.9669    0.9733    0.9701       150
        Tech     0.9597    0.7933    0.8686       150

    accuracy                         0.8830      1000
   macro avg     0.7983    0.7688    0.7809      1000
weighted avg     0.9098    0.8830    0.8937      1000



# CoT

In [ ]:
content = '''أنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال:
1. الثقافة
2. المال
3. الطب
4. السياسة
5. الدين
6. الرياضة
7. التكنولوجيا

اتبع الخطوات التالية عند تحليل محتوى كل مقال:

الخطوة 1: فهم المحتوى الأساسي
اقرأ المقال بالكامل بعناية. حدد الحدث أو القضية أو الرسالة الرئيسية. قد يكون ذلك خبرًا أو تعليقًا أو تقريرًا يتعلق بمجالات مثل السياسة أو الصحة أو الاقتصاد أو الرياضة أو التكنولوجيا أو الثقافة.

الخطوة 2: تحديد الموضوع السائد
حدد الموضوع أو الفكرة الرئيسية. ركز على محتوى المقال والمفاهيم المتكررة فيه. تجاهل المواضيع الثانوية أو الجوانب الجانبية التي لا تُعبّر عن التركيز العام للمقال.

الخطوة 3: مطابقة المقال مع فئة
اختر الفئة الوحيدة التي تتوافق بشكل أفضل مع الموضوع السائد. كن دقيقًا: اختر الفئة الأكثر تحديدًا والأقرب صلة. لا تعتمد على الافتراضات أو المعرفة الخلفية خارج النص. إذا تناول المقال عدة مواضيع، اختر الموضوع الذي يتم التأكيد عليه أكثر أو الذي يُعتبر محوريًا لغرض المقال.

المثال 1
محتوى المقال:
أكد الفنان عزت أبوعوف، رئيس مهرجان القاهرة السينمائي، إلغاء حفل ختام المهرجان الذي كان من المزمع أن يقام غداً، وذلك بسبب الظروف غير المستقرة التي تمر بها مصر حالياً.ونفى أبوعوف في تصريحات لـ"العربية.نت" سفر الضيوف الأجانب خوفاً مما تشهده مصر من مظاهرات، وما تردد عن أن هذا السفر قد تسبب في عدم إمكانية إقامة حفل الختام، مؤكداً أن وزير الثقافة دكتور محمد صابر عرب هو من أصدر قرار الإلغاء.وعن توزيع الجوائز التي كان من المزمع أن يتم في حفل الختام، شرح أنه سيقام مؤتمر صحافي سيكون بديلاً عن الحفل وسيتم من خلاله إعلان توزيع الجوائز والأفلام الفائزة في هذه الدورة.يُذكر أن حفل افتتاح المهرجان أيضاً تم في هدوء وبعيداً عن الصخب المعتاد، وذلك أيضاً بسبب الظروف نفسها التي تشهدها مصر.

أفكار:
- المحتوى الرئيسي: إلغاء حفل ختام مهرجان القاهرة السينمائي
- الثيمة الرئيسية: الأنشطة الثقافية والفنية
- الفئة الأنسب: الثقافة

الفئة المتوقعة: الثقافة

المثال 2
محتوى المقال:
قال الرئيس التنفيذي للشركة السعودية للكهرباء زياد الشيحة، في مقابلة عبر الهاتف مع قناة "العربية"، إنه لأول مرة في تاريخ الشركة تراجع استهلاك المملكة في فترات الذروة. وأضاف الشيحة أن التراجع الذي حدث في استهلاك المملكة في 2016، مقارنة مع عام 2015، دفع الشركة لمراجعة السعات المطلوبة لدى دراسة المشاريع الجديدة. وأكد أن مسألة انخفاض الحمل الذروي لأول مرة في تاريخ الشركة عن العام جلعنا نراجع المحطات المستقبلية والتي ستكون بعقود شراء الطاقة، وهذا سيكون لمشاربع الإنتاج وتتم مراجعة السعات المطلوبة خاصة مع قلة الحمل الذروي في 2016. وستزود الشركة السعودية للكهرباء شركة زين السعودية بشبكة الألياف البصرية الممتدة لـ 60 ألف كم. وأوضح الشيحة أن "قطاع التوليد سيطرح للخصخصة كما هو معلن، ونعمل على الموضوع بشكل متوازن وشبه يومي". وكانت خسائر شركة السعودية للكهرباء قد تفاقمت بأكثر من 60%، في الربع الأخير من العام الماضي، مقارنة بالربع المماثل من عام 2015، لتبلغ 2.34 مليار ريال. من ناحية أخرى، ارتفعت أرباح الشركة بنسبة 37%، خلال العام الماضي، مقارنةً بعام 2015، لتبلغ 2.1 مليار ريال. وأرجعت الشركة تفاقم الخسائر الفصلية إلى ارتفاع تكلفة المبيعات نتيجة الزيادة في أسعار الوقود وارتفاع المصاريف التشغيلية.

أفكار:
- المحتوى الرئيسي: مقابلة مع زياد الشيحة، الرئيس التنفيذي للشركة السعودية للكهرباء
- الثيمة الرئيسية: الاقتصاد والطاقة
- الفئة الأنسب: المال والأعمال

الفئة المتوقعة: المال


المثال 3
محتوى المقال:
يعتقد بعض المدخنين أن السجائر الإلكترونية تعد أحد أهم العوامل المساعدة في الإقلاع عن التدخين، في حين يعتقد البعض الآخر أنها تعتبر وسيلة إغواء للاستمرار في الخضوع لتلك العادة المدمرة، إلا أن الأبحاث الطبية الحديثة تشير إلى أن الأخطار والفوائد لتلك النوعية من السجائر لاتزال غير معلومة بصورة واضحة بين المدخنين، وذلك وفق ما نشرت وكالة أنباء الشرق الأوسط المصرية. وكانت مجموعة من الباحثين قد أجرت أبحاثها على أكثر من 64 مدخنا، ولم ينجحوا في تحقيق إجماع حول الفوائد والأضرار المحتملة للسجائر الإلكترونية، وهو ما قد يعكس انقساما في المجتمع الطبي حول مدى ملاءمة تعزيز السجائر الإلكترونية كبديل أكثر أمنا للتدخين. وأوضح الباحثون أن معظم المشاركين في الدراسة يرون أن التدخين يعتبر شكلا من الإدمان، حيث تلعب الإرادة دورا قويا في الإقلاع عن هذه العادة المدمرة، في الوقت الذى حاول فيه جميع المشاركين في الدراسة مرة واحدة على الأقل الإقلاع عن العادة المدمرة.

أفكار:
- المحتوى الرئيسي: السجائر الإلكترونية ودورها في الإقلاع عن التدخين
- الثيمة الرئيسية: الصحة والطب
- الفئة الأنسب: الطب

الفئة المتوقعة: الطب


المثال 4
محتوى المقال:
أعرب المتحدث باسم الهيئة العليا للمفاوضات السورية سالم المسلط عن أمله في أن تنتقل روسيا فعليا لتقف إلى جانب الشعب السوري بدلا من النظام، وذلك عقب قراره بسحب القوات الروسية من سوريا. وأضاف المسلط أن هناك جدية لمست مؤخرا حيال المواقف الروسية للدفع نحو الحل السياسي للأزمة في سوريا، خلال جولة المحادثات الجديدة التي انطلقت في جنيف. هذا وأعلن متحدث باسم الرئيس الروسي فلاديمير بوتين بوتين بأن روسيا أبلغت الأسد بقرار سحب الجزء الرئيسي من القوات الروسية من سوريا. وقال المتحدث إن بوتين خلال اجتماعه بوزير دفاعه أمر اعتبارا من اليوم (الثلاثاء) ببدء سحب الجزء الرئيسي من القوات الروسية. في حين قال متحدث باسم بوتين إن القاعدة البحرية والجوية الروسية في سوريا تستمر في العمل كما في السابق. ووفقا للكرملين فإن بوتين طلب من وزير خارجيته سيرغي لافروف تكثيف الدور الروسي في عملية السلام في سوريا، مشيرا إلى ان  القوات الروسية في سوريا أوجدت ظروفا ملائمة لعملية السلام.

أفكار:
- المحتوى الرئيسي: تصريحات سالم المسلط، المتحدث باسم الهيئة العليا للمفاوضات السورية
- الثيمة الرئيسية: السياسة والعلاقات الدولية
- الفئة الأنسب: السياسة

الفئة المتوقعة: السياسة


المثال 5
محتوى المقال:
أكد عبداللطيف بخاري، رئيس لجنة الخبراء باتحاد القدم السعودي أن اتحاده يبحث عن 15 منصباً في اللجان الآسيوية، وأن هناك لجنة برئاسة خالد المرزوقي، عضو مجلس إدارة الاتحاد مكلفة بالاختيار. وقال بخاري لـ"في المرمى":" الترشيحات تتم بناء على معايير قارية، ونحن نبحث عن 15 منصباً في لجان الاتحاد الآسيوي". وبين بخاري أن لجنة المسابقات اقترحت إيجاد مراقب لكل مباراة، وزاد:" تقارير المراقب لن تغني عن تقارير الحكم، أما بالنسبة لتقارير الأول فيمكن للجنة الانضباط الاستناد عليها".

أفكار:
- المحتوى الرئيسي: تصريحات عبداللطيف بخاري، رئيس لجنة الخبراء في اتحاد القدم السعودي
- الثيمة الرئيسية: الرياضة
- الفئة الأنسب: الرياضة

الفئة المتوقعة: الرياضة


المثال 6
محتوى المقال:
يبدو أن هاتف #آيفون7  الجديد الذي ستصدره شركة آبل، لن يحمل الكثير من التغيرات، بحسب ما أفاد تقرير لـ "وول ستريت جورنول". فالهاتف الجديد سيأتي شبيها بالنسخة الحالية (آيفون6)، مع تغيير جذري على صعيد "الصوت" والسماعات. وحسب تقرير الصحيفة قد تزيل شركة #آبل منفذ سماعة الصوت، لتدمجه بالمنفذ الذي يوضع فيه شاحن الهاتف في الأسفل. وتراهن آبل من خلال إزالة المنفذ على جعل هاتفها الجديد "أرفع"، ومضاد للماء فإزالة فتحة السماعة، ستمنع تسرب الماء إلى الجهاز عبر الثقب، وتعطيله. ومن شأن إزالة المنفذ الذي يصل قطره إلى 2.5 ميلليمتر أن ينعكس إيجابا أيضاً على البطارية، على اعتبار أن التغيير سيفسح مساحة جديدة يمكن استغلالها. إلا أن التقرير لم يفصل كيف يمكن لمنفذ الشحن الجديد أن يمنع بدوره تسرب الماء إلى داخل الهاتف. في المقابل، يرى بعض منتقدي الشكل الجديد أو التغيير المنتظر أن الاستغناء عن السماعات التقليدية، سيجبر المستخدمين على شراء السماعات الأغلى التي تعمل بتقنية "بلوتوث".

أفكار:
- المحتوى الرئيسي: تقريرًا حول هاتف آيفون 7 الجديد من شركة آبل
- الثيمة الرئيسية: التكنولوجيا
- الفئة الأنسب: التكنولوجيا

الفئة المتوقعة: التكنولوجيا


المثال 7
محتوى المقال:
} قال رسول الله صلى الله عليه وسلم: «من أتى فراشه وهو ينوي أن يقوم يصلي من الليل فغلبته عينه حتى أصبح، كتب له ما نوى، وكان نومه صدقة عليه من ربه».} وقال صلى الله عليه وسلم: «ما تشاور قومإلا هداهم الله لأرشد أمورهم».

أفكار:
- المحتوى الرئيسي: الأحاديث المذكورة هي من أقوال النبي محمد صلى الله عليه وسلم
- الثيمة الرئيسية: الدين والعقيدة
- الفئة الأنسب: الدين

الفئة المتوقعة: الدين
'''

cot_pred = []
for i, text in enumerate(data['content']):
    prompt = f'''المقال الذي عليك تصنيفه
    محتوى المقال:
    {text}'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature = 0.2,
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    cot_pred.append(response)

In [ ]:
pred_cot = pd.DataFrame()
pred_cot['Predicted'] = cot_pred
pred_cot['Predicted'].value_counts()

,count
Predicted,
الرياضة,137
الثقافة,128
التكنولوجيا,112
السياسة,109
الطب,89
...,...
الثيمة الرئيسية: الرياضة\nالفئة الأنسب: الرياضة,1
"المقال يندرج تحت فئة ""التكنولوجيا""، ولكن بالنظر إلى أن المقال يتناول مشكلة تتعلق بسلامة المنتج (هاتف آيفون 7)، يمكن أيضا تصنيفه تحت فئة ""المال"" أو ""الأعمال"" لأن هذا النوع من الحوادث قد يؤثر على سمعة الشركة وربما يؤدي إلى خسائر مالية. ومع ذلك، بناءً على التركيز الرئيسي للمقال، وهو تقنية الهاتف، فإن الفئة الأكثر ملاءمة هي ""التكنولوجيا"". ولكن، إذا تم النظر في التأثير المحتمل على الشركة، يمكن أيضا اعتبار ""المال"" كفئة صحيحة. ولكن بناءً على السؤال، فإن الفئة الأكثر مباشرة هي ""التكنولوجيا"".",1
الفئة المتوقعة: العلم والتكنولوجيا,1


In [ ]:
pred_cot['Article'] = data['content']
pred_cot.to_excel('Fanar-NC-CoT.xlsx', index = False)

The output is checked for normalizing predictions since the output does not follow a single format

In [ ]:
pred_cot = pd.read_excel('Fanar-NC-CoT.xlsx')

In [ ]:
pred_cot['Predicted Normalized'].value_counts()

,count
Predicted Normalized,
المال,169
الثقافة,164
السياسة,160
الرياضة,152
الطب,138
التكنولوجيا,119
الدين,89
Other,9


In [ ]:
nor_pre = []
for pr in pred_cot['Predicted Normalized']:
  if "الرياضة" in pr:
    nor_pre.append("Sports")
  elif "الصحة" in pr:
    nor_pre.append("Medical")
  elif "الطب" in pr:
    nor_pre.append("Medical")
  elif "الثقافة" in pr:
    nor_pre.append("Culture")
  elif "المال" in pr:
    nor_pre.append("Finance")
  elif "السياسة" in pr:
    nor_pre.append("Politics")
  elif "الدين" in pr:
    nor_pre.append("Religion")
  elif "التكنولوجيا" in pr:
    nor_pre.append("Tech")
  else:
    nor_pre.append("Other")

In [ ]:
y_true = data['subdirectory'].values
print(classification_report(y_true, nor_pre, digits = 4))

              precision    recall  f1-score   support

     Culture     0.8110    0.8867    0.8471       150
     Finance     0.7988    0.9000    0.8464       150
     Medical     0.9565    0.8800    0.9167       150
       Other     0.0000    0.0000    0.0000         0
    Politics     0.8313    0.8867    0.8581       150
    Religion     0.9101    0.8100    0.8571       100
      Sports     0.9671    0.9800    0.9735       150
        Tech     0.9580    0.7600    0.8476       150

    accuracy                         0.8750      1000
   macro avg     0.7791    0.7629    0.7683      1000
weighted avg     0.8894    0.8750    0.8791      1000

